In [ ]:
# ==========================================
# THE AGILE WEAPON: AUTOGLUON for Y_LFC_B (Quick Test + Final Model)
# (Pipeline + 3-Year Trend + 5-Fold CV + SMOTENC + Optimal Cut-off + Feature Importance)
# ==========================================

# 1. ติดตั้ง Library
!pip install autogluon.tabular
!pip install autogluon.tabular[catboost]==1.5.0
!pip install imbalanced-learn

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from autogluon.tabular import TabularPredictor
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score, roc_curve, confusion_matrix, classification_report
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer, KNNImputer
from sklearn.preprocessing import RobustScaler
from imblearn.over_sampling import SMOTENC
import os
import shutil
import time
from datetime import datetime
from google.colab import drive

# 2. เตรียมข้อมูล
print("🚀 Starting QUICK AutoGluon for Y_LFC_B...")
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

FILE_PATH = '/content/drive/MyDrive/Thesis/RawData2.xlsx'
TARGET_COL = 'Y_LFC_B'

df = pd.read_excel(FILE_PATH)
df = df.dropna(subset=[TARGET_COL, 'LegalEntity_YN', 'YER'])

# ==========================================================
# 🛑 ส่วนที่ 1: กำหนดFeatures
# ==========================================================
non_fin_features = [
    "PRC_CFW", "ECO_ADT", "POL_BEN", "SAV_PDPA", "CAP_NETW",
    "OHR_ORG", "BEH_MON"
]

cols_67 = [
    "2567_Profitability_NetIncomePerTotalRevenue%",
    "2567_LiquidityRatio_CurrentRatio",
    "2567_LeverageRatio_TotalLiabilityPerEquity"
]
cols_66 = [
    "2566_Profitability_NetIncomePerTotalRevenue%",
    "2566_LiquidityRatio_CurrentRatio",
    "2566_LeverageRatio_TotalLiabilityPerEquity"
]
cols_65 = [
    "2565_Profitability_NetIncomePerTotalRevenue%",
    "2565_LiquidityRatio_CurrentRatio",
    "2565_LeverageRatio_TotalLiabilityPerEquity"
]
financial_ratio_cols = cols_67 + cols_66 + cols_65

feature_cols = [col for col in non_fin_features + financial_ratio_cols if col in df.columns]
data = df[feature_cols + ['YER', 'LegalEntity_YN', TARGET_COL]].copy()

# ==========================================================
# 🛑 ส่วนที่ 2: PRE-PIPELINE HANDLING
# ==========================================================
data = data.replace(['', ' ', 'NA', 'N/A', 'NaN'], np.nan)
mask_legal_no = data['LegalEntity_YN'] == 'ไม่ใช่'
data.loc[mask_legal_no, financial_ratio_cols] = np.nan
mask_legal_yes = data['LegalEntity_YN'] == 'ใช่'
data.loc[mask_legal_yes & (data['YER'] == 0), financial_ratio_cols] = np.nan
data.loc[mask_legal_yes & (data['YER'] == 1), cols_66 + cols_65] = np.nan
data.loc[mask_legal_yes & (data['YER'] == 2), cols_65] = np.nan
data = data.drop(columns=['LegalEntity_YN'], errors='ignore')

# ==========================================================
# 🛑 ส่วนที่ 3: OPTIMIZED CUSTOM TRANSFORMER PIPELINE
# ==========================================================
def optimized_preprocess(X_train, X_test, yers_train, yers_test):
    X_tr = X_train.copy()
    X_te = X_test.copy()

    # 1. Non-Fin Features (Median)
    train_medians = X_tr[non_fin_features].median()
    X_tr[non_fin_features] = X_tr[non_fin_features].fillna(train_medians)
    X_te[non_fin_features] = X_te[non_fin_features].fillna(train_medians)

    # 2. Financial Ratios (Winsorize 5-95 & Robust Scaler)
    scaler = RobustScaler()
    for col in financial_ratio_cols:
        if col not in X_tr.columns: continue

        if col in cols_67: mask_tr, mask_te = yers_train >= 1, yers_test >= 1
        elif col in cols_66: mask_tr, mask_te = yers_train >= 2, yers_test >= 2
        elif col in cols_65: mask_tr, mask_te = yers_train >= 3, yers_test >= 3

        if mask_tr.any():
            lower_bound, upper_bound = X_tr.loc[mask_tr, col].quantile([0.05, 0.95])
            X_tr.loc[mask_tr, col] = X_tr.loc[mask_tr, col].clip(lower=lower_bound, upper=upper_bound)
            X_te.loc[mask_te, col] = X_te.loc[mask_te, col].clip(lower=lower_bound, upper=upper_bound)

        train_vals = X_tr.loc[mask_tr & X_tr[col].notnull(), col].values.reshape(-1, 1)
        test_vals = X_te.loc[mask_te & X_te[col].notnull(), col].values.reshape(-1, 1)

        if len(train_vals) > 0:
            X_tr.loc[mask_tr & X_tr[col].notnull(), col] = scaler.fit_transform(train_vals).flatten()
            if len(test_vals) > 0:
                X_te.loc[mask_te & X_te[col].notnull(), col] = scaler.transform(test_vals).flatten()

    # 3. Row-wise Median
    def fill_row_median(df_mod, yers):
        for index, row in df_mod.iterrows():
            yer_val = yers.loc[index]
            if yer_val >= 1:
                valid_cols = []
                if yer_val >= 1: valid_cols.extend(cols_67)
                if yer_val >= 2: valid_cols.extend(cols_66)
                if yer_val >= 3: valid_cols.extend(cols_65)

                valid_cols = [c for c in valid_cols if c in df_mod.columns]
                row_median = row[valid_cols].median()
                if pd.notna(row_median):
                    df_mod.loc[index, valid_cols] = row[valid_cols].fillna(row_median)
        return df_mod

    X_tr = fill_row_median(X_tr, yers_train)
    X_te = fill_row_median(X_te, yers_test)

    # 4. KNN Imputation
    knn = KNNImputer(n_neighbors=5)
    mask_tr_knn = yers_train > 0
    if mask_tr_knn.sum() > 0:
        imputed_train = knn.fit_transform(X_tr.loc[mask_tr_knn, financial_ratio_cols])
        X_tr.loc[mask_tr_knn, financial_ratio_cols] = imputed_train

    mask_te_knn = yers_test > 0
    if mask_te_knn.sum() > 0:
        imputed_test = knn.transform(X_te.loc[mask_te_knn, financial_ratio_cols])
        X_te.loc[mask_te_knn, financial_ratio_cols] = imputed_test

    # 5. บังคับกลับเป็น NaNตาม YER
    for df_mod, yers in [(X_tr, yers_train), (X_te, yers_test)]:
        df_mod.loc[yers == 0, financial_ratio_cols] = np.nan
        df_mod.loc[yers == 1, [c for c in cols_66 + cols_65 if c in financial_ratio_cols]] = np.nan
        df_mod.loc[yers == 2, [c for c in cols_65 if c in financial_ratio_cols]] = np.nan

    # 🌟 FEATURE AGGREGATION (3-Year Trend)
    ratio_groups = {
        'Avg_3Y_NetIncomePerTotalRevenue': [
            "2567_Profitability_NetIncomePerTotalRevenue%",
            "2566_Profitability_NetIncomePerTotalRevenue%",
            "2565_Profitability_NetIncomePerTotalRevenue%"
        ],
        'Avg_3Y_CurrentRatio': [
            "2567_LiquidityRatio_CurrentRatio",
            "2566_LiquidityRatio_CurrentRatio",
            "2565_LiquidityRatio_CurrentRatio"
        ],
        'Avg_3Y_TotalLiabilityPerEquity': [
            "2567_LeverageRatio_TotalLiabilityPerEquity",
            "2566_LeverageRatio_TotalLiabilityPerEquity",
            "2565_LeverageRatio_TotalLiabilityPerEquity"
        ]
    }

    new_features_list = []
    for new_col, cols_to_avg in ratio_groups.items():
        cols_to_avg = [c for c in cols_to_avg if c in financial_ratio_cols]
        new_features_list.append(new_col)
        for df_mod in [X_tr, X_te]:
            if cols_to_avg:
                df_mod[new_col] = df_mod[cols_to_avg].mean(axis=1)

    cols_to_drop = [col for col in financial_ratio_cols if col.startswith('2565') or col.startswith('2566')]
    X_tr.drop(columns=cols_to_drop, inplace=True, errors='ignore')
    X_te.drop(columns=cols_to_drop, inplace=True, errors='ignore')

    return X_tr, X_te, new_features_list

# ==========================================================
# 🛑 ฟังก์ชันประเมินผล OOF (Quick 5-Fold)
# ==========================================================
def evaluate_pipeline_oof(data_df, feature_list, target_col):
    X = data_df[feature_list + ['YER']]
    y = data_df[target_col]

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    acc_scores, auc_scores, f1_scores = [], [], []
    oof_predictions = np.zeros(len(y))

    print("\n⏳เริ่มต้น 5-Fold CV (High Quality Mode, 5 นาที/รอบ)...")
    start_time = time.time()

    for fold, (train_idx, test_idx) in enumerate(skf.split(X, y)):
        fold_start = time.time()
        X_train_raw, X_test_raw = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        X_train_clean, X_test_clean, _ = optimized_preprocess(
            X_train_raw.drop(columns=['YER']), X_test_raw.drop(columns=['YER']), X_train_raw['YER'], X_test_raw['YER']
        )

        train_data_fold = X_train_clean.copy()

        # --- SAFE SMOTENC ---
        imputer_for_smote = IterativeImputer(random_state=42, max_iter=5)
        X_train_imputed = pd.DataFrame(imputer_for_smote.fit_transform(train_data_fold), columns=train_data_fold.columns)

        cat_indices = [X_train_imputed.columns.get_loc(col) for col in non_fin_features if col in X_train_imputed.columns]
        smote_nc = SMOTENC(categorical_features=cat_indices, random_state=42)
        X_resampled, y_resampled = smote_nc.fit_resample(X_train_imputed, y_train.values)

        train_data_fold_resampled = pd.DataFrame(X_resampled, columns=X_train_clean.columns)

        # บังคับกลับสู่กฎ YER
        for idx in train_data_fold.index:
            if idx in train_data_fold_resampled.index:
                for col in train_data_fold.columns:
                    if pd.isna(train_data_fold.loc[idx, col]):
                        train_data_fold_resampled.loc[idx, col] = np.nan

        train_data_fold_resampled[target_col] = y_resampled

        # Train Model
        predictor = TabularPredictor(label=target_col, eval_metric='roc_auc', problem_type='binary', verbosity=0).fit(
            train_data_fold_resampled, presets='high_quality', time_limit=300
        )

        y_prob = predictor.predict_proba(X_test_clean).iloc[:, 1]

        fpr, tpr, thresholds = roc_curve(y_test, y_prob)
        optimal_idx = np.argmax(tpr - fpr)
        optimal_threshold = thresholds[optimal_idx]
        y_pred_optimal = (y_prob >= optimal_threshold).astype(int)

        oof_predictions[test_idx] = y_pred_optimal.values
        acc_scores.append(accuracy_score(y_test, y_pred_optimal))
        auc_scores.append(roc_auc_score(y_test, y_prob))
        f1_scores.append(f1_score(y_test, y_pred_optimal))

        fold_time = time.time() - fold_start
        print(f"  รอบที่ {fold+1}/5 เสร็จสิ้น | AUC: {auc_scores[-1]:.4f} | เวลา: {fold_time/60:.1f}นาที")

    total_time = time.time() - start_time
    print(f"✅ เสร็จสมบูรณ์! ใช้เวลาทั้งหมด: {total_time/60:.1f} นาที")
    return np.mean(acc_scores), np.std(acc_scores), np.mean(auc_scores), np.std(auc_scores), np.mean(f1_scores), oof_predictions

# รันประเมินผลแบบด่วน
mean_acc, sd_acc, mean_auc, sd_auc, mean_f1, y_pred_oof = evaluate_pipeline_oof(data, feature_cols, TARGET_COL)

print("\n" + "="*80)
print(f"📊 QUICK VERDICT for {TARGET_COL} (5-Fold CV + SMOTENC + Optimal Threshold)")
print("="*80)
print(f"Accuracy  : {mean_acc:.4f} (S.D. {sd_acc:.4f})")
print(f"ROC-AUC   : {mean_auc:.4f} (S.D. {sd_auc:.4f})")
print(f"F1-Score  : {mean_f1:.4f}")
print("="*80)

# ==========================================================
# 🛑 4. สร้าง Final Model สำหรับดู Feature Importance
# ==========================================================
print("\n⏳ กำลังสร้าง Final Model (High Quality) สำหรับดู Feature Importance...")
X_final, _, _ = optimized_preprocess(data[feature_cols], data[feature_cols], data['YER'], data['YER'])

final_train_data = X_final.copy()
final_train_data[TARGET_COL] = data[TARGET_COL]

predictor_final = TabularPredictor(label=TARGET_COL, eval_metric='roc_auc', problem_type='binary', verbosity=0).fit(
    final_train_data, presets='high_quality', time_limit=300)

print("\n📊 Confusion Matrix (Out-of-Fold)...")
cm = confusion_matrix(data[TARGET_COL], y_pred_oof)
plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Oranges', xticklabels=['No(0)', 'Yes(1)'], yticklabels=['No(0)', 'Yes(1)'], annot_kws={"size": 16})
plt.title(f'Confusion Matrix (OOF) for {TARGET_COL}', fontsize=16, pad=15)
plt.ylabel('Actual', fontsize=12)
plt.xlabel('Predicted', fontsize=12)
plt.show()

print(f"\n📋 Classification Report (OOF):")
print(classification_report(data[TARGET_COL], y_pred_oof))

fi_main = predictor_final.feature_importance(final_train_data)
top_10 = fi_main.head(10)

print(f"\n🌟 Top 10 Features 🌟")
print(top_10)

plt.figure(figsize=(10, 6))
sns.barplot(x=top_10['importance'], y=top_10.index, hue=top_10.index, palette='magma', legend=False)
plt.title(f'Top 10 Feature Importance ({TARGET_COL})', fontsize=16)
plt.xlabel('Importance Score', fontsize=12)
plt.ylabel('Features', fontsize=12)
plt.tight_layout()
plt.show()

print("\n📊 สรุปอันดับตัวแปรทั้งหมดที่มีผลต่อการพยากรณ์")
print(fi_main.to_string())

print("\n📦 Freezing Main Model to Google Drive...")
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
output_filename = f'/content/drive/MyDrive/Thesis/Final_AutoGluon_YLFC_B_AgileWeapon_{timestamp}'
shutil.make_archive(output_filename, 'zip', predictor_final.path)
print(f"✅ Saved successfully: {output_filename}.zip")